# 호우날두 — 피처 묶음 비교 실험 (feature_exp v1)
`hou_baseline_v1.ipynb`의 STEP 0–2, `make_features`, 누수 자가검증을 **그대로 가져와서** 그 위에 실험만 추가한 노트북.

| 실험 | 추가 피처 (베이스라인에 없는 것만) |
|---|---|
| 기준 | 베이스라인 피처 그대로 |
| 1 | 수위 lag·최근 변화량: 시간별 변화 시퀀스, 2·4·5시간 변화량, 최근 최대 시간당 상승, 연속 상승 시간 |
| 2 | 최근 구간 누적 강우: 2·3·12시간 누적, 13–24시간 구간 강우 |
| 3 | 수위·강우 이동평균: 지수가중 이동평균(EWM) |
| 최종 | 1–3 중 효과 있었던 묶음 조합 |

**비교 조건**: 모든 실험이 같은 fold·파라미터·seed. 저장 항목은 전체 OOF RMSE, 기준 대비 변화량, fold별 개선 여부, 사상별 RMSE, 실행 시간.

**해석 원칙**: 변화량은 "기준 피처 위에 이 묶음을 더했을 때의 개선 효과"로 기록. 피처끼리 정보가 겹치므로 개별 변수의 독립 기여로 해석하지 않음.

**검증 데이터**: 튜닝·선택 과정의 낙관성은 같은 fold 재검증으로는 없어지지 않으므로, 실험 전에 train 사상 일부를 홀드아웃으로 떼어두고 **최종 조합 확정 후 한 번만** 확인.

## STEP 0. 환경 설정

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

# 레포 루트 또는 notebooks/ 어디서 실행해도 data/를 찾음
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_DIR = ROOT / "data"
OUT_DIR = DATA_DIR.parent / "outputs"
OUT_DIR.mkdir(exist_ok=True)

SEED = 42
N_FOLDS = 4
TARGET = "target_maxrise_6h"
EXPECTED = {"train_rows": 749_657, "test_rows": 186_263, "train_events": 122, "test_events": 30}
import time, json

RESULT_DIR = OUT_DIR / "feature_exp_v1"
RESULT_DIR.mkdir(exist_ok=True)

USE_HOLDOUT = True       # 실험 전에 train 사상 일부를 떼어 최종 확인용으로 보관
HOLDOUT_FRAC = 0.15      # 122개 중 약 18개 사상
MIN_GAIN = 0.0005        # 이 이상 RMSE가 줄고
MIN_FOLDS_IMPROVED = 3   # 4개 fold 중 3개 이상에서 개선돼야 '효과 있음'으로 판정

## STEP 1. 데이터 로드

In [ ]:
def find_file(stem):
    hits = sorted(p for p in DATA_DIR.glob(stem + "*") if not p.name.startswith("~$"))
    if not hits:
        raise FileNotFoundError(f"{stem} 파일을 {DATA_DIR}에서 찾지 못함")
    return hits[0]

def read_table(path):
    if path.suffix in (".xlsx", ".xls"):
        return pd.read_excel(path)
    for enc in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩 확인 필요: {path}")

train = pd.read_csv(DATA_DIR / "train.csv.gz")
test = pd.read_csv(DATA_DIR / "test.csv.gz")
sub = read_table(find_file("sample_submission"))
ddict = read_table(find_file("data_dictionary"))

for df in (train, test):
    df["timestamp"] = pd.to_datetime(df["timestamp"])

print("train:", train.shape, "| test:", test.shape, "| sub:", sub.shape)
print("행 수 일치:", len(train) == EXPECTED["train_rows"], len(test) == EXPECTED["test_rows"])
print("test 관측소 중 train에 없는 관측소:", len(set(test.station_id) - set(train.station_id)))
ddict

## STEP 2. 호우사상 ID 복원

In [ ]:
def add_event_id(df, prefix, gap_h=1):
    ts = pd.Series(np.sort(df["timestamp"].unique()))
    event_no = ts.diff().gt(pd.Timedelta(hours=gap_h)).cumsum()
    m = pd.DataFrame({"timestamp": ts, "event_id": prefix + event_no.astype(str).str.zfill(3)})
    return df.merge(m, on="timestamp", how="left")

train = add_event_id(train, "TR_E")
test = add_event_id(test, "TE_E")

print("train 사상 수:", train.event_id.nunique(), "(기대", EXPECTED["train_events"], ")")
print("test  사상 수:", test.event_id.nunique(), "(기대", EXPECTED["test_events"], ")")
# 개수가 다르면 사상 사이 간격이 1시간인 경우가 있다는 뜻 → gap_h 조정 또는 wl 연속성으로 추가 검증 필요

## STEP 3. 홀드아웃 사상 분리
- 사상 단위로 무작위 분리 (seed 고정). 이후 모든 실험·선택은 **dev 사상만** 사용
- 관측소 정규화 통계도 dev 사상으로만 계산
- 홀드아웃이 빠지므로 CV 점수는 베이스라인 노트북(122개 사상 전체)과 정확히 같지 않음 → 이 노트북 안에서의 비교만 의미 있음

In [ ]:
def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))

events = np.sort(train["event_id"].unique())
rng = np.random.default_rng(SEED)
if USE_HOLDOUT:
    n_hold = int(round(len(events) * HOLDOUT_FRAC))
    hold_events = set(rng.choice(events, size=n_hold, replace=False))
else:
    hold_events = set()

is_hold = train["event_id"].isin(hold_events)
print(f"dev 사상 {len(events) - len(hold_events)}개 / 홀드아웃 사상 {len(hold_events)}개")
print(f"dev 행 {(~is_hold).sum():,} / 홀드아웃 행 {is_hold.sum():,}")
pd.Series(sorted(hold_events), name="event_id").to_csv(RESULT_DIR / "holdout_events.csv", index=False)

## STEP 4. 베이스라인 피처 (베이스라인 노트북과 동일한 함수)

In [ ]:
EPS = 1e-6

def make_features(df, station_stats):
    df = df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)
    g = df.groupby(["station_id", "event_id"], sort=False)

    def lag(col, k):
        # 정확히 k시간 전 행일 때만 사용 (결측 시각이 있으면 NaN)
        ok = (df["timestamp"] - g["timestamp"].shift(k)) == pd.Timedelta(hours=k)
        return g[col].shift(k).where(ok)

    # A. 수위 변화
    for h in (1, 3, 6, 12, 24):
        df[f"dwl_{h}h"] = df["wl_t"] - df[f"wl_t_minus_{h}h"]
    df["accel_wl"] = df["dwl_1h"] - (df["wl_t_minus_1h"] - df["wl_t_minus_3h"]) / 2
    wl_l2, wl_l4, wl_l5 = lag("wl_t", 2), lag("wl_t", 4), lag("wl_t", 5)
    past_max = pd.concat([df["wl_t"], df["wl_t_minus_1h"], wl_l2, df["wl_t_minus_3h"], wl_l4, wl_l5], axis=1).max(axis=1)
    df["past_maxrise_6h"] = past_max - df["wl_t_minus_6h"]      # 직전 6시간 실현 상승량 (t 이하 값만)
    df["wl_below_past_max"] = df["wl_t"] - past_max             # 최근 고점 대비 위치

    # B. 강우 이력
    for c in ("imerg_rain_1h", "sfc_rain_1h", "radar_reflectivity_mean", "radar_reflectivity_ge20_fraction"):
        for k in (1, 2, 3):
            df[f"{c}_lag{k}"] = lag(c, k)
    df["imerg_trend_1h"] = df["imerg_rain_1h"] - df["imerg_rain_1h_lag1"]
    df["sfc_trend_1h"] = df["sfc_rain_1h"] - df["sfc_rain_1h_lag1"]
    for src in ("imerg", "sfc"):
        df[f"{src}_front_1_6"] = df[f"{src}_rain_1h"] / (df[f"{src}_rain_6h"] + EPS)
        df[f"{src}_front_6_24"] = df[f"{src}_rain_6h"] / (df[f"{src}_rain_24h"] + EPS)
        df[f"{src}_rain_7_24h"] = df[f"{src}_rain_24h"] - df[f"{src}_rain_6h"]
        df[f"{src}_event_cum"] = g[f"{src}_rain_1h"].cumsum()
        df[f"{src}_event_max1h"] = g[f"{src}_rain_1h"].cummax()
    df["imerg_minus_sfc_6h"] = df["imerg_rain_6h"] - df["sfc_rain_6h"]

    # C. 사상 맥락
    df["event_elapsed_h"] = (df["timestamp"] - df.groupby("event_id")["timestamp"].transform("min")).dt.total_seconds() / 3600
    df["wl_since_event_start"] = df["wl_t"] - g["wl_t"].transform("first")

    # D. 동시각 공간 (같은 사상·같은 시각)
    gt = df.groupby(["event_id", "timestamp"])
    for c in ("imerg_rain_1h", "imerg_rain_6h", "sfc_rain_6h", "dwl_1h", "dwl_3h", "dwl_6h"):
        df[f"xs_mean_{c}"] = gt[c].transform("mean")
    df["xs_rising_frac"] = gt["dwl_1h"].transform(lambda s: (s > 0).mean())
    df["dwl_3h_vs_xs"] = df["dwl_3h"] - df["xs_mean_dwl_3h"]

    # E. 관측소 정규화 (train 통계)
    df = df.merge(station_stats, on="station_id", how="left")
    df["wl_rel"] = (df["wl_t"] - df["st_wl_med"]) / (df["st_wl_iqr"] + EPS)
    for h in (1, 3, 6):
        df[f"dwl_{h}h_scaled"] = df[f"dwl_{h}h"] / (df["st_wl_iqr"] + EPS)
    return df

dev_raw = train[~is_hold]
q = dev_raw.groupby("station_id")["wl_t"].quantile([0.25, 0.5, 0.75]).unstack()
station_stats = pd.DataFrame({
    "station_id": q.index,
    "st_wl_med": q[0.5].values,
    "st_wl_iqr": (q[0.75] - q[0.25]).values,
})

tr = make_features(train, station_stats)
te = make_features(test, station_stats)
BASE_FEATURES = [c for c in tr.columns if c not in {"row_id", "timestamp", "event_id", TARGET}]
print("베이스라인 피처 수:", len(BASE_FEATURES))

## STEP 5. 실험 피처 생성
베이스라인에 이미 있는 항목(`dwl_1/3/6/12/24h`, `past_maxrise_6h`, 강우 1h lag1–3, 사상 누적·최대 강우, 7–24h 강우)은 다시 만들지 않음.

- **실험 1 (수위)**: 베이스라인은 2·4·5시간 전 수위를 `past_maxrise_6h` 계산에만 썼음 → 시간별 변화 시퀀스로 풀어서 추가
- **실험 2 (누적 강우)**: 주어진 1·6·24h 사이의 빈 구간(2·3·12h)을 이전 행에서 계산
- **실험 3 (이동평균)**: 단순 이동평균은 실험 2의 누적값을 시간으로 나눈 것과 같아 중복 → 최근 값에 가중을 더 주는 **EWM**(반감기 3h·12h)으로 구성

In [ ]:
def add_exp_features(df):
    df = df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)
    keys = ["station_id", "event_id"]
    g = df.groupby(keys, sort=False)

    def lag(col, k):
        ok = (df["timestamp"] - g["timestamp"].shift(k)) == pd.Timedelta(hours=k)
        return g[col].shift(k).where(ok)

    new = {}

    # 실험 1: 수위 lag·최근 변화량
    wl = {0: df["wl_t"], 1: df["wl_t_minus_1h"], 2: lag("wl_t", 2), 3: df["wl_t_minus_3h"],
          4: lag("wl_t", 4), 5: lag("wl_t", 5), 6: df["wl_t_minus_6h"]}
    steps = {k: wl[k] - wl[k + 1] for k in range(6)}            # k시간 전의 1시간 변화량
    for k in range(1, 6):
        new[f"x1_step_t{k}"] = steps[k]
    for h in (2, 4, 5):
        new[f"x1_dwl_{h}h"] = wl[0] - wl[h]
    step_df = pd.concat([steps[k] for k in range(6)], axis=1)
    new["x1_max_step_6h"] = step_df.max(axis=1)
    rising = step_df.gt(0).astype(int).values
    streak = np.zeros(len(df), dtype=int)
    alive = np.ones(len(df), dtype=bool)
    for k in range(6):                                            # 현재부터 거꾸로 연속 상승 시간
        alive &= rising[:, k].astype(bool)
        streak += alive
    new["x1_rise_streak"] = streak

    # 실험 2: 최근 구간 누적 강우
    for src in ("imerg", "sfc"):
        r1 = df[f"{src}_rain_1h"]
        l1, l2 = lag(f"{src}_rain_1h", 1), lag(f"{src}_rain_1h", 2)
        new[f"x2_{src}_rain_2h"] = r1 + l1
        new[f"x2_{src}_rain_3h"] = r1 + l1 + l2
        rain_12h = df[f"{src}_rain_6h"] + lag(f"{src}_rain_6h", 6)
        new[f"x2_{src}_rain_12h"] = rain_12h
        new[f"x2_{src}_rain_13_24h"] = df[f"{src}_rain_24h"] - rain_12h
    new["x2_radar_mean_3h"] = pd.concat(
        [df["radar_reflectivity_mean"], lag("radar_reflectivity_mean", 1), lag("radar_reflectivity_mean", 2)], axis=1
    ).mean(axis=1)

    # 실험 3: 지수가중 이동평균 (현재 행까지만 사용)
    ewm_src = df[keys + ["imerg_rain_1h", "sfc_rain_1h", "radar_reflectivity_mean", "dwl_1h"]]
    for hl in (3, 12):
        e = (ewm_src.groupby(keys, sort=False)[["imerg_rain_1h", "sfc_rain_1h", "radar_reflectivity_mean", "dwl_1h"]]
             .ewm(halflife=hl).mean().reset_index(level=[0, 1], drop=True).sort_index())
        new[f"x3_imerg_ewm{hl}"] = e["imerg_rain_1h"]
        new[f"x3_sfc_ewm{hl}"] = e["sfc_rain_1h"]
        new[f"x3_radar_ewm{hl}"] = e["radar_reflectivity_mean"]
        new[f"x3_dwl1_ewm{hl}"] = e["dwl_1h"]

    return pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)

tr = add_exp_features(tr)
te = add_exp_features(te)

GROUPS = {
    "1_수위변화": [c for c in tr.columns if c.startswith("x1_")],
    "2_누적강우": [c for c in tr.columns if c.startswith("x2_")],
    "3_이동평균": [c for c in tr.columns if c.startswith("x3_")],
}
dup = set(sum(GROUPS.values(), [])) & set(BASE_FEATURES)
print("베이스라인과 겹치는 이름:", dup if dup else "없음")
for k, v in GROUPS.items():
    print(f"{k}: {len(v)}개 -> {v}")

### 5-1. 누수 자가검증 (베이스라인 + 실험 피처 전체)
베이스라인 노트북 STEP 4-1과 같은 방식: 각 사상의 마지막 6시간 행을 지우고 피처를 다시 만든 뒤, 남은 행의 값이 같은지 비교.

동시각 전체 관측소 평균(`xs_*`)은 Q&A 10/6 (나) "동일 호우사상 다른 관측소의 시점 t 정보"로 대회 규정상 허용. 이 검증은 **미래 행 참조 여부**만 확인하므로, 실제 운영 시 모든 관측소 자료가 같은 시각에 들어온다는 가정은 방법론 문서에 별도로 명시.

In [ ]:
end = train.groupby("event_id")["timestamp"].transform("max")
cut = train[train["timestamp"] <= end - pd.Timedelta(hours=6)]
tr_cut = add_exp_features(make_features(cut, station_stats)).set_index("row_id")
tr_full = tr.set_index("row_id").loc[tr_cut.index]

check_cols = [c for c in tr_cut.columns if c not in ("timestamp", "event_id", "station_id", TARGET)]
diff = {c: int((~np.isclose(tr_full[c].astype(float), tr_cut[c].astype(float), equal_nan=True)).sum())
        for c in check_cols}
leaky = {c: n for c, n in diff.items() if n > 0}
print(f"검사한 피처 {len(check_cols)}개 | 미래 의존 피처:", leaky if leaky else "없음 ✅")

## STEP 6. 고정 실험 조건

In [ ]:
cats = pd.CategoricalDtype(sorted(train.station_id.unique()))
tr["station_id"] = tr["station_id"].astype(cats)
te["station_id"] = te["station_id"].astype(cats)

dev = tr[~tr["event_id"].isin(hold_events)].reset_index(drop=True)
hold = tr[tr["event_id"].isin(hold_events)].reset_index(drop=True)

PARAMS = dict(
    objective="regression", learning_rate=0.05, num_leaves=63,
    min_data_in_leaf=200, feature_fraction=0.8, bagging_fraction=0.8,
    bagging_freq=1, lambda_l2=1.0, seed=SEED, verbose=-1,
)
FOLDS = list(GroupKFold(n_splits=N_FOLDS).split(dev, dev[TARGET], dev["event_id"]))  # 모든 실험이 이 fold 공유

def run_cv(features, name):
    t0 = time.time()
    oof = np.zeros(len(dev))
    fold_rmse, best_iters = [], []
    for i_tr, i_va in FOLDS:
        d_tr = lgb.Dataset(dev.iloc[i_tr][features], dev.iloc[i_tr][TARGET])
        d_va = lgb.Dataset(dev.iloc[i_va][features], dev.iloc[i_va][TARGET])
        m = lgb.train(PARAMS, d_tr, num_boost_round=5000, valid_sets=[d_va],
                      callbacks=[lgb.early_stopping(200, verbose=False)])
        oof[i_va] = m.predict(dev.iloc[i_va][features], num_iteration=m.best_iteration)
        fold_rmse.append(rmse(dev.iloc[i_va][TARGET], oof[i_va]))
        best_iters.append(m.best_iteration)
    res = {
        "name": name, "features": features, "oof": oof,
        "oof_rmse": rmse(dev[TARGET], oof), "fold_rmse": fold_rmse,
        "best_iters": best_iters, "runtime_sec": round(time.time() - t0, 1),
    }
    pd.DataFrame({"row_id": dev["row_id"], "oof": oof}).to_csv(RESULT_DIR / f"oof_{name}.csv", index=False)
    print(f"[{name}] 피처 {len(features)}개 | OOF RMSE {res['oof_rmse']:.4f} | "
          f"fold {np.round(fold_rmse, 4).tolist()} | {res['runtime_sec']}s")
    return res

## STEP 7. 실험 실행 (기준 → 1 → 2 → 3)

In [ ]:
results = {"기준": run_cv(BASE_FEATURES, "기준")}
for gname, cols in GROUPS.items():
    results[gname] = run_cv(BASE_FEATURES + cols, gname)

### 7-1. 최종 조합
기준 대비 OOF RMSE가 `MIN_GAIN` 이상 줄고, 4개 fold 중 `MIN_FOLDS_IMPROVED`개 이상에서 개선된 묶음만 조합.

In [ ]:
def compare(res, ref):
    gain = ref["oof_rmse"] - res["oof_rmse"]
    folds_up = int(sum(a < b for a, b in zip(res["fold_rmse"], ref["fold_rmse"])))
    return gain, folds_up

selected = []
for gname in GROUPS:
    gain, folds_up = compare(results[gname], results["기준"])
    ok = gain >= MIN_GAIN and folds_up >= MIN_FOLDS_IMPROVED
    print(f"{gname}: 개선 {gain:+.4f} | 개선 fold {folds_up}/{N_FOLDS} -> {'채택' if ok else '보류'}")
    if ok:
        selected.append(gname)

if len(selected) >= 2:
    final_cols = BASE_FEATURES + sum((GROUPS[g] for g in selected), [])
    results["최종"] = run_cv(final_cols, "최종")
elif len(selected) == 1:
    results["최종"] = {**results[selected[0]], "name": "최종"}
    print("채택 묶음이 1개라 해당 실험 결과를 최종으로 사용")
else:
    results["최종"] = {**results["기준"], "name": "최종"}
    print("채택 묶음 없음 -> 기준 유지")
print("최종 조합:", ["기준"] + selected)

### 7-2. 결과 표 · 사상별 오차 저장

In [ ]:
rows = []
for name, r in results.items():
    gain, folds_up = compare(r, results["기준"])
    rows.append({
        "실험": name, "피처 수": len(r["features"]),
        "OOF RMSE": round(r["oof_rmse"], 4), "기준 대비 변화(음수=개선)": round(-gain, 4) + 0.0,
        "개선 fold 수": folds_up if name != "기준" else None,
        "fold RMSE": np.round(r["fold_rmse"], 4).tolist(),
        "실행 시간(s)": r["runtime_sec"],
    })
summary = pd.DataFrame(rows)
summary.to_csv(RESULT_DIR / "experiment_summary.csv", index=False, encoding="utf-8-sig")

ev_rows = []
for name, r in results.items():
    e = (dev.assign(oof=r["oof"]).groupby("event_id")
         .apply(lambda d: pd.Series({"rmse": rmse(d[TARGET], d["oof"]),
                                     "sse": float(((d[TARGET] - d["oof"]) ** 2).sum()),
                                     "n_rows": len(d)})))
    ev_rows.append(e.assign(실험=name).reset_index())
event_table = pd.concat(ev_rows, ignore_index=True)
event_table.to_csv(RESULT_DIR / "event_rmse.csv", index=False, encoding="utf-8-sig")
summary

### 7-3. 사상별 오차 확인
"소수 대형 사상이 Global RMSE를 좌우하는가"를 우리 OOF로 직접 확인. 오차제곱합 비중이 크지 않으면 특정 사상 맞춤 개선보다 전반적 개선이 우선.

In [ ]:
piv_rmse = event_table.pivot(index="event_id", columns="실험", values="rmse")
base_sse = event_table[event_table["실험"] == "기준"].set_index("event_id")["sse"].sort_values(ascending=False)

share = base_sse.cumsum() / base_sse.sum()
print(f"기준 모델: 상위 5개 사상 오차제곱합 비중 {share.iloc[4]:.1%}, 상위 10개 {share.iloc[min(9, len(share)-1)]:.1%}")
print(f"사상별 RMSE: 중앙값 {piv_rmse['기준'].median():.3f}, 최댓값 {piv_rmse['기준'].max():.3f}")

top = base_sse.head(10).index
view = piv_rmse.loc[top].round(4)
view["최종-기준"] = (view["최종"] - view["기준"]).round(4)
print("\n오차 상위 10개 사상의 실험별 RMSE")
display(view)

rest = piv_rmse.drop(index=top)
print("나머지 사상 평균 RMSE 변화 (최종-기준):", round((rest["최종"] - rest["기준"]).mean(), 4))

## STEP 8. 홀드아웃 확인 (최종 조합 확정 후 **한 번만** 실행)
- dev 전체로 학습, 반복 수는 CV의 best_iteration 평균 사용 (홀드아웃으로 early stopping 하지 않음)
- 기준과 최종을 같은 조건으로 비교. 이 결과를 보고 피처를 다시 고르면 홀드아웃도 낙관적으로 변하므로 재선택에 쓰지 않음

In [ ]:
RUN_HOLDOUT = False   # 최종 조합 확정 후 True로 바꿔 한 번 실행

if RUN_HOLDOUT and len(hold):
    hold_rows = []
    for name in ("기준", "최종"):
        r = results[name]
        n_round = int(np.mean(r["best_iters"]))
        m = lgb.train(PARAMS, lgb.Dataset(dev[r["features"]], dev[TARGET]), num_boost_round=n_round)
        p = m.predict(hold[r["features"]])
        hold_rows.append({"실험": name, "홀드아웃 RMSE": round(rmse(hold[TARGET], p), 4),
                          "CV OOF RMSE": round(r["oof_rmse"], 4), "반복 수": n_round})
    hold_summary = pd.DataFrame(hold_rows)
    hold_summary.to_csv(RESULT_DIR / "holdout_check.csv", index=False, encoding="utf-8-sig")
    display(hold_summary)

## STEP 9. 한동에게 넘길 최종 피처 목록 저장

In [ ]:
final_info = {
    "selected_groups": selected,
    "features": results["최종"]["features"],
    "params": PARAMS,
    "holdout_events": sorted(hold_events),
    "note": "변화량은 기준 피처 위에 묶음을 더했을 때의 효과. 개별 변수의 독립 기여 아님.",
}
with open(RESULT_DIR / "final_features.json", "w", encoding="utf-8") as f:
    json.dump(final_info, f, ensure_ascii=False, indent=2)
print("저장 폴더:", RESULT_DIR)
print(sorted(p.name for p in RESULT_DIR.iterdir()))